# KELT-20b TESS Exploration With mlexo

This notebook keeps the original `lightkurve` exploration flow from `kelt20b-tess.ipynb`, but replaces the old PyMC / exoplanet and JAX modeling path with your local `mlexo` code from `/Volumes/drive-2tb/code/mlexo`.

Notes:
- The TESS download, normalization, and detrending steps remain CPU / `lightkurve` code.
- The transit forward model comes from the local `mlexo` checkout.
- The fit here is deterministic (`scipy.optimize.minimize`) because the local `mlexo` package currently exposes forward-model pieces, not a sampler.
- The bootstrap cell below loads the local package directly from disk and aliases its current internal `mlxoplanet` imports.
        


In [ ]:
import warnings

import lightkurve as lk
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning, module="pytensor")

TARGET = "KELT-20"
MISSION = "TESS"
AUTHOR = "SPOC"
EXPTIME_S = 120

PERIOD_D = 3.4741039
T0_BTJD = 1698.210775
TRANSIT_DURATION_D = 3.54 / 24.0
PLOT_WINDOW_D = 0.15
MODEL_WINDOW_D = PLOT_WINDOW_D
RADIUS_RATIO_GUESS = 0.116
B_GUESS = 0.60
LC_ORDER = 20
EXPOSURE_TIME_D = EXPTIME_S / 86400.0

search = lk.search_lightcurve(TARGET, mission=MISSION, author=AUTHOR, exptime=EXPTIME_S)
display(search)

lc_collection = search.download_all()
print(f"Downloaded {len(lc_collection)} light curves")
        


In [ ]:
stitched_lc = lc_collection.stitch().remove_nans().normalize()
transit_mask = stitched_lc.create_transit_mask(
    period=PERIOD_D,
    transit_time=T0_BTJD,
    duration=TRANSIT_DURATION_D,
)
flat_lc = stitched_lc.flatten(window_length=901, mask=transit_mask)
folded_lc = flat_lc.fold(period=PERIOD_D, epoch_time=T0_BTJD)
binned_folded_lc = folded_lc.bin(time_bin_size=5.0 / (24.0 * 60.0))

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=False)

stitched_lc.scatter(ax=axes[0], color="0.35", alpha=0.35, s=4)
axes[0].set_title("KELT-20 stitched TESS light curve")
axes[0].set_ylabel("Normalized flux")

folded_lc.scatter(ax=axes[1], color="0.6", alpha=0.18, s=6, label="Unbinned")
binned_folded_lc.scatter(ax=axes[1], color="tab:blue", s=18, label="5 min bins")
axes[1].set_xlim(-PLOT_WINDOW_D, PLOT_WINDOW_D)
axes[1].set_title("Phase-folded transit")
axes[1].set_xlabel("Time from mid-transit [days]")
axes[1].set_ylabel("Normalized flux")
axes[1].legend()

plt.tight_layout()
        


In [ ]:
sector_lc = lc_collection[0].remove_nans().normalize()
sector_transit_mask = sector_lc.create_transit_mask(
    period=PERIOD_D,
    transit_time=T0_BTJD,
    duration=TRANSIT_DURATION_D,
)
sector_flat_lc = sector_lc.flatten(window_length=901, mask=sector_transit_mask)
sector_folded_lc = sector_flat_lc.fold(period=PERIOD_D, epoch_time=T0_BTJD)

fig, ax = plt.subplots(figsize=(10, 5))
sector_folded_lc.plot_river(ax=ax)
ax.set_aspect("auto")
ax.set_title("KELT-20b river plot for the first downloaded TESS sector")
plt.tight_layout()
        


In [ ]:
time_btjd = flat_lc.time.value
phase_days = ((time_btjd - T0_BTJD + 0.5 * PERIOD_D) % PERIOD_D) - 0.5 * PERIOD_D
epoch_number = np.round((time_btjd - T0_BTJD) / PERIOD_D).astype(int)

fig, ax = plt.subplots(figsize=(11, 6))
unique_epochs = np.unique(epoch_number)
colors = plt.cm.tab20(np.linspace(0, 1, len(unique_epochs)))

for color, current_epoch in zip(colors, unique_epochs):
    mask = (epoch_number == current_epoch) & (np.abs(phase_days) < PLOT_WINDOW_D)
    if not np.any(mask):
        continue
    ax.errorbar(
        phase_days[mask],
        flat_lc.flux.value[mask],
        yerr=flat_lc.flux_err.value[mask],
        fmt="o",
        ms=4,
        color=color,
        ecolor="0.75",
        elinewidth=0.8,
        capsize=0,
        alpha=0.85,
        label=f"Orbit {current_epoch}",
    )

ax.set_title("Epoch-to-epoch transit overlay")
ax.set_xlabel("Time from mid-transit [days]")
ax.set_ylabel("Normalized flux")
ax.grid(alpha=0.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8, ncol=1)
plt.tight_layout()
        


## Joint Multi-Sector Transit + Per-Sector GP Model With Local mlexo

The next cells keep the shared Keplerian transit model from your local `mlexo` checkout, but add an independent exact GP to each TESS sector. The GP is block-diagonal across sectors, so there is no covariance across long inter-sector gaps.
        


In [ ]:
def build_joint_tess_dataset(lc_collection, period_d, t0_btjd, transit_duration_d, model_window_d):
    sector_labels = []
    x_parts = []
    y_parts = []
    yerr_parts = []
    sector_idx_parts = []

    for sector_number, raw_lc in enumerate(lc_collection, start=1):
        sector_label = raw_lc.meta.get("MISSION", f"sector_{sector_number}")
        sector_lc = raw_lc.remove_nans().normalize()
        transit_mask = sector_lc.create_transit_mask(
            period=period_d,
            transit_time=t0_btjd,
            duration=transit_duration_d,
        )
        _, oot_outlier_mask = sector_lc[~transit_mask].remove_outliers(sigma=5.0, return_mask=True)
        keep_mask = np.ones(len(sector_lc), dtype=bool)
        oot_rows = np.flatnonzero(~transit_mask)
        keep_mask[oot_rows[oot_outlier_mask]] = False
        sector_lc = sector_lc[keep_mask]
        transit_mask = sector_lc.create_transit_mask(
            period=period_d,
            transit_time=t0_btjd,
            duration=transit_duration_d,
        )
        sector_flat = sector_lc.flatten(window_length=901, mask=transit_mask)

        sector_phase = ((sector_flat.time.value - t0_btjd + 0.5 * period_d) % period_d) - 0.5 * period_d
        window_mask = np.abs(sector_phase) < model_window_d
        if not np.any(window_mask):
            continue

        flux = np.asarray(sector_flat.flux.value, dtype=np.float64)
        flux_err = np.asarray(sector_flat.flux_err.value, dtype=np.float64)

        finite_flux = flux[np.isfinite(flux)]
        flux_fill = float(np.median(finite_flux)) if finite_flux.size else 1.0
        flux = np.where(np.isfinite(flux), flux, flux_fill)

        finite_err = flux_err[np.isfinite(flux_err) & (flux_err > 0)]
        err_fill = float(np.median(finite_err)) if finite_err.size else 5.0e-4
        flux_err = np.where(np.isfinite(flux_err) & (flux_err > 0), flux_err, err_fill)

        x_parts.append(np.ascontiguousarray(sector_flat.time.value[window_mask], dtype=np.float64))
        y_parts.append(np.ascontiguousarray(flux[window_mask] - 1.0, dtype=np.float64))
        yerr_parts.append(np.ascontiguousarray(flux_err[window_mask], dtype=np.float64))
        sector_idx_parts.append(np.full(np.count_nonzero(window_mask), len(sector_labels), dtype=np.int32))
        sector_labels.append(sector_label)

    if not x_parts:
        raise RuntimeError("No TESS cadences survived the preprocessing window.")

    x = np.concatenate(x_parts)
    y = np.concatenate(y_parts)
    yerr = np.concatenate(yerr_parts)
    sector_idx = np.concatenate(sector_idx_parts)

    sort_idx = np.argsort(x)
    x = x[sort_idx]
    y = y[sort_idx]
    yerr = yerr[sort_idx]
    sector_idx = sector_idx[sort_idx]

    n_sectors = len(sector_labels)
    sector_counts = np.bincount(sector_idx, minlength=n_sectors)
    sector_rows = [np.flatnonzero(sector_idx == i) for i in range(n_sectors)]

    return {
        "time": x,
        "flux": y,
        "flux_err": yerr,
        "sector_idx": sector_idx,
        "sector_labels": sector_labels,
        "sector_counts": sector_counts,
        "sector_rows": sector_rows,
        "n_sectors": n_sectors,
    }


dataset = build_joint_tess_dataset(
    lc_collection=lc_collection,
    period_d=PERIOD_D,
    t0_btjd=T0_BTJD,
    transit_duration_d=TRANSIT_DURATION_D,
    model_window_d=MODEL_WINDOW_D,
)

print(f"Prepared {dataset['time'].size} cadences across {dataset['n_sectors']} sectors")
for label, count in zip(dataset["sector_labels"], dataset["sector_counts"]):
    print(f"  {label}: {int(count)} cadences in the transit window")
        


In [ ]:
import importlib.util
import sys
from pathlib import Path

import emcee
from scipy.optimize import minimize

MLEXO_ROOT = Path("/Volumes/drive-2tb/code/mlexo")
if not MLEXO_ROOT.exists():
    raise FileNotFoundError(f"Local mlexo checkout not found: {MLEXO_ROOT}")

if "mlxoplanet" not in sys.modules:
    spec = importlib.util.spec_from_file_location(
        "mlxoplanet",
        MLEXO_ROOT / "__init__.py",
        submodule_search_locations=[str(MLEXO_ROOT)],
    )
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load package spec from {MLEXO_ROOT}")
    mlxoplanet = importlib.util.module_from_spec(spec)
    sys.modules["mlxoplanet"] = mlxoplanet
    spec.loader.exec_module(mlxoplanet)
else:
    mlxoplanet = sys.modules["mlxoplanet"]

sys.modules.setdefault("mlexo", mlxoplanet)

import mlx.core as mx
from mlxoplanet import constants as mlexo_constants
from mlxoplanet.light_curves import limb_dark_light_curve, transforms as lc_transforms
from mlxoplanet.orbits import Body, Central, OrbitalBody

RHO_SUN_UNIT = 3.0 / (4.0 * np.pi)
RHO_STAR_SOLAR_GUESS = 0.3938
RHO_STAR_SOLAR_SIGMA = 0.10
GP_RHO_GUESS_D = 0.05
GP_RHO_MIN_D = max(5.0 * EXPOSURE_TIME_D, 0.01)
GP_RHO_MAX_D = 0.30
GP_EPS = 1.0e-9
GP_MX_DTYPE = mx.float32
EMCEE_NWALKERS_MIN = 48
EMCEE_BURNIN_STEPS = 150
EMCEE_PRODUCTION_STEPS = 300
EMCEE_THIN = 5
EMCEE_INIT_SCALE = 1.0e-3

mlexo = sys.modules["mlexo"]
print("Loaded local mlexo from", MLEXO_ROOT)
        


In [ ]:
def kipping_q_to_u(q1, q2):
    sqrt_q1 = np.sqrt(np.clip(q1, 0.0, 1.0))
    u1 = 2.0 * sqrt_q1 * q2
    u2 = sqrt_q1 * (1.0 - 2.0 * q2)
    return np.asarray([u1, u2], dtype=np.float64)


def rho_star_solar_to_a_over_rstar(period, rho_star_solar):
    period = np.asarray(period, dtype=np.float64)
    rho_star_solar = np.asarray(rho_star_solar, dtype=np.float64)
    rho_star_unit = rho_star_solar * RHO_SUN_UNIT
    return ((mlexo_constants.G * period**2 * rho_star_unit) / (3.0 * np.pi)) ** (1.0 / 3.0)


def circular_transit_duration(period, a_over_rstar, b, r):
    chord = np.sqrt(np.clip((1.0 + r) ** 2 - b**2, 0.0, None))
    sin_i = np.sqrt(np.clip(1.0 - (b / a_over_rstar) ** 2, 1.0e-12, 1.0))
    arg = np.clip(chord / (a_over_rstar * sin_i), 0.0, 1.0)
    return (period / np.pi) * np.arcsin(arg)


def matern32_kernel_mx(time_mx, gp_sigma_mx, gp_rho_mx):
    dt = mx.abs(time_mx[:, None] - time_mx[None, :])
    sqrt3 = mx.sqrt(mx.array(3.0, dtype=time_mx.dtype))
    scaled = sqrt3 * dt / gp_rho_mx
    return gp_sigma_mx**2 * (1.0 + scaled) * mx.exp(-scaled)


def gp_cholesky_solve(cov, rhs):
    chol = mx.linalg.cholesky(cov, stream=mx.cpu)
    y = mx.linalg.solve_triangular(chol, rhs, upper=False, stream=mx.cpu)
    alpha = mx.linalg.solve_triangular(mx.transpose(chol), y, upper=True, stream=mx.cpu)
    return chol, alpha


def sector_gp_marginal_nll(time, resid, flux_err, gp_sigma, gp_rho, jitter, eps=GP_EPS):
    time_mx = mx.array(time, dtype=GP_MX_DTYPE)
    resid_mx = mx.array(resid, dtype=GP_MX_DTYPE)[:, None]
    flux_err_mx = mx.array(flux_err, dtype=GP_MX_DTYPE)
    gp_sigma_mx = mx.array(gp_sigma, dtype=GP_MX_DTYPE)
    gp_rho_mx = mx.array(gp_rho, dtype=GP_MX_DTYPE)
    jitter_mx = mx.array(jitter, dtype=GP_MX_DTYPE)

    k_signal = matern32_kernel_mx(time_mx, gp_sigma_mx, gp_rho_mx)
    diag = flux_err_mx**2 + jitter_mx**2 + mx.array(eps, dtype=GP_MX_DTYPE)
    cov = mx.array(k_signal + mx.diag(diag), dtype=GP_MX_DTYPE)

    chol, alpha = gp_cholesky_solve(cov, resid_mx)
    quad = mx.sum(resid_mx * alpha)
    logdet = 2.0 * mx.sum(mx.log(mx.diag(chol)))
    n_data = float(time.shape[0])
    nll = 0.5 * (quad + logdet + n_data * np.log(2.0 * np.pi))
    return float(np.asarray(nll))


def sector_gp_posterior_mean(time, resid, flux_err, gp_sigma, gp_rho, jitter, eps=GP_EPS):
    time_mx = mx.array(time, dtype=GP_MX_DTYPE)
    resid_mx = mx.array(resid, dtype=GP_MX_DTYPE)[:, None]
    flux_err_mx = mx.array(flux_err, dtype=GP_MX_DTYPE)
    gp_sigma_mx = mx.array(gp_sigma, dtype=GP_MX_DTYPE)
    gp_rho_mx = mx.array(gp_rho, dtype=GP_MX_DTYPE)
    jitter_mx = mx.array(jitter, dtype=GP_MX_DTYPE)

    k_signal = matern32_kernel_mx(time_mx, gp_sigma_mx, gp_rho_mx)
    diag = flux_err_mx**2 + jitter_mx**2 + mx.array(eps, dtype=GP_MX_DTYPE)
    cov = mx.array(k_signal + mx.diag(diag), dtype=GP_MX_DTYPE)
    _, alpha = gp_cholesky_solve(cov, resid_mx)
    gp_mean = k_signal @ alpha
    return np.asarray(gp_mean, dtype=np.float64).reshape(-1)


def unpack_params(theta, n_sectors):
    theta = np.asarray(theta, dtype=np.float64)
    period = np.exp(theta[0])
    t0 = theta[1]
    rho_star_solar = np.exp(theta[2])
    r = np.exp(theta[3])
    b_scaled = theta[4]
    q1 = theta[5]
    q2 = theta[6]

    mean_flux = theta[7 : 7 + n_sectors]
    log_jitter = theta[7 + n_sectors : 7 + 2 * n_sectors]
    log_gp_sigma = theta[7 + 2 * n_sectors : 7 + 3 * n_sectors]
    log_gp_rho = theta[7 + 3 * n_sectors : 7 + 4 * n_sectors]

    jitter = np.exp(log_jitter)
    gp_sigma = np.exp(log_gp_sigma)
    gp_rho = np.exp(log_gp_rho)

    b = b_scaled * (1.0 + r)
    a_over_rstar = rho_star_solar_to_a_over_rstar(period, rho_star_solar)
    cos_i = np.clip(b / a_over_rstar, -1.0, 1.0)
    duration = circular_transit_duration(period, a_over_rstar, b, r)

    return {
        "period": period,
        "t0": t0,
        "rho_star_solar": rho_star_solar,
        "rho_star_unit": rho_star_solar * RHO_SUN_UNIT,
        "a_over_rstar": a_over_rstar,
        "duration": duration,
        "inclination_deg": np.degrees(np.arccos(cos_i)),
        "r": r,
        "b": b,
        "q1": q1,
        "q2": q2,
        "u": kipping_q_to_u(q1, q2),
        "mean_flux": mean_flux,
        "log_jitter": log_jitter,
        "jitter": jitter,
        "log_gp_sigma": log_gp_sigma,
        "gp_sigma": gp_sigma,
        "log_gp_rho": log_gp_rho,
        "gp_rho": gp_rho,
    }


def mlexo_transit_flux(time, period, t0, a_over_rstar, b, r, u, exposure_time_d=EXPOSURE_TIME_D, lc_order=LC_ORDER):
    central = Central.from_orbital_properties(
        period=period,
        semimajor=a_over_rstar,
        radius=1.0,
    )
    body = Body(
        period=period,
        time_transit=t0,
        impact_param=b,
        radius=r,
    )
    orbit = OrbitalBody(central, body)
    flux_func = limb_dark_light_curve(orbit, u, order=lc_order)
    if exposure_time_d and exposure_time_d > 0:
        flux_func = lc_transforms.integrate(
            flux_func,
            exposure_time=exposure_time_d,
            order=1,
            num_samples=7,
        )
    flux = flux_func(mx.array(time))
    return np.asarray(flux, dtype=np.float64)


def build_gp_trend(time, flux, flux_err, sector_rows, transit_model, pars):
    gp_trend = np.zeros_like(flux)
    for sector_id, rows in enumerate(sector_rows):
        baseline = transit_model[rows] + pars["mean_flux"][sector_id]
        resid = flux[rows] - baseline
        gp_trend[rows] = sector_gp_posterior_mean(
            time=time[rows],
            resid=resid,
            flux_err=flux_err[rows],
            gp_sigma=pars["gp_sigma"][sector_id],
            gp_rho=pars["gp_rho"][sector_id],
            jitter=pars["jitter"][sector_id],
        )
    return gp_trend


def negative_log_posterior(theta, time, flux, flux_err, sector_rows, n_sectors):
    pars = unpack_params(theta, n_sectors)
    transit_model = mlexo_transit_flux(
        time=time,
        period=pars["period"],
        t0=pars["t0"],
        a_over_rstar=pars["a_over_rstar"],
        b=pars["b"],
        r=pars["r"],
        u=pars["u"],
    )

    nll = 0.0
    for sector_id, rows in enumerate(sector_rows):
        baseline = transit_model[rows] + pars["mean_flux"][sector_id]
        resid = flux[rows] - baseline
        nll += sector_gp_marginal_nll(
            time=time[rows],
            resid=resid,
            flux_err=flux_err[rows],
            gp_sigma=pars["gp_sigma"][sector_id],
            gp_rho=pars["gp_rho"][sector_id],
            jitter=pars["jitter"][sector_id],
        )

    flux_err_ref = max(float(np.median(flux_err)), 1.0e-6)
    prior = 0.0
    prior += 0.5 * ((pars["period"] - PERIOD_D) / 2.0e-4) ** 2
    prior += 0.5 * ((pars["t0"] - T0_BTJD) / 3.0e-3) ** 2
    prior += 0.5 * ((pars["rho_star_solar"] - RHO_STAR_SOLAR_GUESS) / RHO_STAR_SOLAR_SIGMA) ** 2
    prior += 0.5 * ((pars["r"] - RADIUS_RATIO_GUESS) / 0.03) ** 2
    prior += 0.5 * ((pars["b"] - B_GUESS) / 0.25) ** 2
    prior += 0.5 * np.sum((pars["mean_flux"] / 5.0e-4) ** 2)
    prior += 0.5 * np.sum(((pars["log_jitter"] - np.log(flux_err_ref)) / 2.0) ** 2)
    prior += 0.5 * np.sum(((pars["log_gp_sigma"] - np.log(flux_err_ref)) / 2.0) ** 2)
    prior += 0.5 * np.sum(((pars["log_gp_rho"] - np.log(GP_RHO_GUESS_D)) / 1.0) ** 2)

    return float(nll + prior)


def summarize_interval(samples):
    samples = np.asarray(samples, dtype=np.float64).reshape(-1)
    q16, q50, q84 = np.percentile(samples, [16.0, 50.0, 84.0])
    return {
        "lower": float(q16),
        "median": float(q50),
        "upper": float(q84),
        "plus": float(q84 - q50),
        "minus": float(q50 - q16),
    }


def build_bounds(n_sectors):
    bounds = [
        (np.log(PERIOD_D - 0.01), np.log(PERIOD_D + 0.01)),
        (T0_BTJD - 0.05, T0_BTJD + 0.05),
        (np.log(0.05), np.log(3.0)),
        (np.log(0.01), np.log(0.25)),
        (0.0, 1.0),
        (1.0e-6, 1.0),
        (1.0e-6, 1.0),
    ]
    bounds.extend([(-5.0e-3, 5.0e-3)] * n_sectors)
    bounds.extend([(np.log(1.0e-7), np.log(5.0e-3))] * n_sectors)
    bounds.extend([(np.log(1.0e-7), np.log(5.0e-3))] * n_sectors)
    bounds.extend([(np.log(GP_RHO_MIN_D), np.log(GP_RHO_MAX_D))] * n_sectors)
    return np.asarray(bounds, dtype=np.float64)


def in_bounds(theta, bounds):
    theta = np.asarray(theta, dtype=np.float64)
    return np.all(theta >= bounds[:, 0]) and np.all(theta <= bounds[:, 1])


def log_probability(theta, time, flux, flux_err, sector_rows, n_sectors, bounds):
    theta = np.asarray(theta, dtype=np.float64)
    if not np.all(np.isfinite(theta)) or not in_bounds(theta, bounds):
        return -np.inf

    try:
        nlp = negative_log_posterior(theta, time, flux, flux_err, sector_rows, n_sectors)
    except Exception:
        return -np.inf

    if not np.isfinite(nlp):
        return -np.inf
    return -float(nlp)


def initialize_walkers(theta_map, bounds, n_walkers, seed=123, init_scale=EMCEE_INIT_SCALE):
    theta_map = np.asarray(theta_map, dtype=np.float64)
    bounds = np.asarray(bounds, dtype=np.float64)
    widths = bounds[:, 1] - bounds[:, 0]
    eps = np.maximum(1.0e-8, 1.0e-6 * widths)
    lower = bounds[:, 0] + eps
    upper = bounds[:, 1] - eps

    rng = np.random.default_rng(seed)
    walkers = theta_map + init_scale * widths * rng.standard_normal((n_walkers, theta_map.size))
    walkers = np.clip(walkers, lower, upper)

    for idx in range(n_walkers):
        if np.allclose(walkers[idx], theta_map):
            walkers[idx] = np.clip(theta_map + eps * rng.standard_normal(theta_map.size), lower, upper)
    return walkers


def summarize_posterior_samples(theta_samples, n_sectors):
    theta_samples = np.asarray(theta_samples, dtype=np.float64)
    if theta_samples.size == 0:
        return None

    keys = ["r", "b", "rho_star_solar", "a_over_rstar", "inclination_deg", "duration"]
    derived = {key: [] for key in keys}
    depth_percent = []

    for theta in theta_samples:
        pars = unpack_params(theta, n_sectors)
        for key in keys:
            derived[key].append(pars[key])
        depth_percent.append(100.0 * pars["r"] ** 2)

    summary = {key: summarize_interval(values) for key, values in derived.items()}
    summary["transit_depth_percent"] = summarize_interval(depth_percent)
    return summary
        


In [ ]:
gp_sigma_guess = max(float(np.median(dataset["flux_err"])), 1.0e-6)
bounds = build_bounds(dataset["n_sectors"])

theta0 = np.concatenate(
    [
        np.array([
            np.log(PERIOD_D),
            T0_BTJD,
            np.log(RHO_STAR_SOLAR_GUESS),
            np.log(RADIUS_RATIO_GUESS),
            B_GUESS / (1.0 + RADIUS_RATIO_GUESS),
            0.25,
            0.25,
        ], dtype=np.float64),
        np.zeros(dataset["n_sectors"], dtype=np.float64),
        np.full(dataset["n_sectors"], np.log(gp_sigma_guess), dtype=np.float64),
        np.full(dataset["n_sectors"], np.log(gp_sigma_guess), dtype=np.float64),
        np.full(dataset["n_sectors"], np.log(GP_RHO_GUESS_D), dtype=np.float64),
    ]
)

result = minimize(
    negative_log_posterior,
    theta0,
    args=(
        dataset["time"],
        dataset["flux"],
        dataset["flux_err"],
        dataset["sector_rows"],
        dataset["n_sectors"],
    ),
    method="L-BFGS-B",
    bounds=bounds,
    options={"maxiter": 1000, "maxfun": 20000},
)

if not result.success:
    print("Optimizer warning:", result.message)

ndim = result.x.size
n_walkers = max(EMCEE_NWALKERS_MIN, 2 * ndim)
initial_walkers = initialize_walkers(result.x, bounds, n_walkers, seed=123, init_scale=EMCEE_INIT_SCALE)

sampler = emcee.EnsembleSampler(
    n_walkers,
    ndim,
    log_probability,
    args=(
        dataset["time"],
        dataset["flux"],
        dataset["flux_err"],
        dataset["sector_rows"],
        dataset["n_sectors"],
        bounds,
    ),
)

state = sampler.run_mcmc(initial_walkers, EMCEE_BURNIN_STEPS, progress=True)
sampler.reset()
sampler.run_mcmc(state, EMCEE_PRODUCTION_STEPS, progress=True)
posterior_theta = sampler.get_chain(flat=True, thin=EMCEE_THIN)
posterior_log_prob = sampler.get_log_prob(flat=True, thin=EMCEE_THIN)

finite_mask = np.isfinite(posterior_log_prob)
posterior_theta = posterior_theta[finite_mask]
posterior_log_prob = posterior_log_prob[finite_mask]
if posterior_theta.size == 0:
    raise RuntimeError("emcee produced no finite posterior samples.")

best_idx = int(np.argmax(posterior_log_prob))
theta_best = posterior_theta[best_idx]

best_fit = unpack_params(theta_best, dataset["n_sectors"])
best_fit["transit_model"] = mlexo_transit_flux(
    time=dataset["time"],
    period=best_fit["period"],
    t0=best_fit["t0"],
    a_over_rstar=best_fit["a_over_rstar"],
    b=best_fit["b"],
    r=best_fit["r"],
    u=best_fit["u"],
)
best_fit["gp_trend"] = build_gp_trend(
    time=dataset["time"],
    flux=dataset["flux"],
    flux_err=dataset["flux_err"],
    sector_rows=dataset["sector_rows"],
    transit_model=best_fit["transit_model"],
    pars=best_fit,
)
best_fit["mean_model"] = best_fit["transit_model"] + best_fit["mean_flux"][dataset["sector_idx"]] + best_fit["gp_trend"]
best_fit["detrended_flux"] = dataset["flux"] - best_fit["mean_flux"][dataset["sector_idx"]] - best_fit["gp_trend"]
best_fit["residual_flux"] = dataset["flux"] - best_fit["mean_model"]
best_fit["phase_days"] = ((dataset["time"] - best_fit["t0"] + 0.5 * best_fit["period"]) % best_fit["period"]) - 0.5 * best_fit["period"]
best_fit["phase_grid"] = np.linspace(-PLOT_WINDOW_D, PLOT_WINDOW_D, 1500)
best_fit["phase_model_grid"] = mlexo_transit_flux(
    time=best_fit["t0"] + best_fit["phase_grid"],
    period=best_fit["period"],
    t0=best_fit["t0"],
    a_over_rstar=best_fit["a_over_rstar"],
    b=best_fit["b"],
    r=best_fit["r"],
    u=best_fit["u"],
)
best_fit["posterior_theta"] = posterior_theta
best_fit["posterior_log_prob"] = posterior_log_prob
best_fit["summary_stats"] = summarize_posterior_samples(posterior_theta, dataset["n_sectors"])
best_fit["map_result"] = result
best_fit["emcee_acceptance_fraction"] = float(np.mean(sampler.acceptance_fraction))

print({
    "period": best_fit["period"],
    "t0": best_fit["t0"],
    "rho_star_solar": best_fit["rho_star_solar"],
    "a_over_rstar": best_fit["a_over_rstar"],
    "duration": best_fit["duration"],
    "r": best_fit["r"],
    "b": best_fit["b"],
    "median_gp_sigma": float(np.median(best_fit["gp_sigma"])),
    "median_gp_rho": float(np.median(best_fit["gp_rho"])),
    "posterior_samples": int(posterior_theta.shape[0]),
    "acceptance_fraction": best_fit["emcee_acceptance_fraction"],
})
        


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 11), sharex=True)

axes[0].plot(dataset["time"], dataset["flux"], ".", color="0.4", alpha=0.22, ms=4, label="Data")
axes[0].plot(dataset["time"], best_fit["mean_model"], color="tab:red", lw=2, label="Transit + sector offset + GP")
axes[0].set_title("Joint multi-sector raw light curve fit")
axes[0].set_ylabel("Flux - 1")
axes[0].legend()

axes[1].plot(dataset["time"], best_fit["detrended_flux"], ".", color="0.35", alpha=0.22, ms=4, label="Offset + GP detrended data")
axes[1].plot(dataset["time"], best_fit["transit_model"], color="tab:blue", lw=2, label="Transit model")
axes[1].set_title("Offset + GP detrended light curve")
axes[1].set_ylabel("Detrended flux")
axes[1].legend()

axes[2].plot(dataset["time"], best_fit["residual_flux"], ".", color="0.4", alpha=0.25, ms=4)
axes[2].axhline(0.0, color="tab:red", lw=1.5)
axes[2].set_title("Residuals")
axes[2].set_xlabel("Time [BTJD]")
axes[2].set_ylabel("Residual flux")

plt.tight_layout()

print("Best-fit sector systematics:")
for label, offset, jitter, gp_sigma, gp_rho in zip(
    dataset["sector_labels"],
    best_fit["mean_flux"],
    best_fit["jitter"],
    best_fit["gp_sigma"],
    best_fit["gp_rho"],
):
    print(
        f"  {label}: offset={offset:+.6e}, jitter={jitter:.6e}, "
        f"gp_sigma={gp_sigma:.6e}, gp_rho={gp_rho:.5f} d"
    )
        


In [ ]:
phase_days = best_fit["phase_days"]

bin_edges = np.linspace(-PLOT_WINDOW_D, PLOT_WINDOW_D, 81)
bin_index = np.digitize(phase_days, bin_edges)
binned_phase = []
binned_flux = []
for idx in range(1, len(bin_edges)):
    mask = bin_index == idx
    if np.count_nonzero(mask) < 3:
        continue
    binned_phase.append(np.mean(phase_days[mask]))
    binned_flux.append(np.mean(best_fit["detrended_flux"][mask]))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(phase_days, best_fit["detrended_flux"], ".", color="0.5", alpha=0.18, ms=4, label="All detrended cadences")
ax.plot(np.asarray(binned_phase), np.asarray(binned_flux), "o", color="tab:blue", ms=5, label="Binned data")
ax.plot(best_fit["phase_grid"], best_fit["phase_model_grid"], color="tab:red", lw=2, label="Transit model")
ax.set_xlim(-PLOT_WINDOW_D, PLOT_WINDOW_D)
ax.set_title("Phase-folded joint multi-sector transit")
ax.set_xlabel("Time from mid-transit [days]")
ax.set_ylabel("Detrended flux")
ax.legend()
plt.tight_layout()

summary_stats = best_fit.get("summary_stats")

def print_interval(label, value, fmt, stats=None):
    if stats is None:
        print(f"{label} = {format(value, fmt)}")
        return
    print(
        f"{label} = {format(stats['median'], fmt)} "
        f"[{format(stats['lower'], fmt)}, {format(stats['upper'], fmt)}]"
    )

print_interval("Rp/R*", best_fit["r"], ".5f", None if summary_stats is None else summary_stats["r"])
print_interval("b", best_fit["b"], ".4f", None if summary_stats is None else summary_stats["b"])
print_interval("rho_star / rho_sun", best_fit["rho_star_solar"], ".4f", None if summary_stats is None else summary_stats["rho_star_solar"])
print_interval("a/R*", best_fit["a_over_rstar"], ".4f", None if summary_stats is None else summary_stats["a_over_rstar"])
print_interval("Inclination [deg]", best_fit["inclination_deg"], ".3f", None if summary_stats is None else summary_stats["inclination_deg"])
print_interval("Duration [d]", best_fit["duration"], ".5f", None if summary_stats is None else summary_stats["duration"])
print_interval("Transit depth (%)", 100.0 * best_fit["r"] ** 2, ".4f", None if summary_stats is None else summary_stats["transit_depth_percent"])
if summary_stats is not None:
    print("Bounds are posterior 16th/50th/84th percentiles from emcee samples.")
    print(f"Acceptance fraction = {best_fit['emcee_acceptance_fraction']:.3f}")
print("Sectors fit:")
for label, count in zip(dataset["sector_labels"], dataset["sector_counts"]):
    print(f"  {label}: {int(count)} cadences")
        
